## 0. Imports and Setup

In [17]:
import sys
import importlib
import logging
from pathlib import Path

import pandas as pd

# VS Code Jupyter sets CWD to the project root, so '.' is the right path.
PROJECT_ROOT = str(Path('.').resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Force reload so source changes are picked up without a kernel restart.
import src.data.injury_loader as _il_mod
importlib.reload(_il_mod)

from src.data.injury_loader import (
    fetch_il_transactions,
    parse_injury_type,
    compute_days_lost,
    build_injury_database,
    load_injury_database,
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('notebook')
print('injury_loader loaded from:', _il_mod.__file__)

injury_loader loaded from: /Users/nateseluga/Pitcher-Injury-Risk/src/data/injury_loader.py


## 1. Configuration

In [18]:
START_YEAR = 2015
END_YEAR   = 2024

# Flip to False for the full 2015–2024 pull.
TEST_MODE  = True
TEST_YEAR  = 2023

INJURIES_DIR = Path('data/raw/injuries')
INJURIES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output dir : {INJURIES_DIR.resolve()}')
print(f'Test mode  : {TEST_MODE}')

Output dir : /Users/nateseluga/Pitcher-Injury-Risk/data/raw/injuries
Test mode  : True


## 2. Load Pitcher ID Filter

The transactions API returns every player — position players, coaches, two-way
players. We restrict to pitchers using the MLBAM IDs collected in notebook 01.
Any player who ever appeared as `pitcher` in Statcast is included.

In [19]:
meta_path = Path('data/raw/player_metadata/pitchers.parquet')
if not meta_path.exists():
    raise FileNotFoundError(
        'Run notebook 01 first to generate data/raw/player_metadata/pitchers.parquet'
    )

pitcher_meta = pd.read_parquet(meta_path)
pitcher_ids  = set(pitcher_meta['player_id'].dropna().astype(int))
print(f'Pitcher filter: {len(pitcher_ids):,} unique MLBAM IDs')

Pitcher filter: 400 unique MLBAM IDs


## 3. Fetch Raw IL Transactions

Queries the MLB Stats API one month at a time. Each row is a single transaction
event — either a placement (IL) or an activation (IA). We keep both; pairing
happens in section 5.

In [20]:
if TEST_MODE:
    il_raw = fetch_il_transactions(TEST_YEAR, TEST_YEAR)
else:
    il_raw = fetch_il_transactions(START_YEAR, END_YEAR)

print(f'\nTotal IL transaction records : {len(il_raw):,}')
print(f'Transaction type breakdown:')
print(il_raw['transaction_type'].value_counts().to_string())
print(f'\nDate range: {il_raw["transaction_date"].min().date()} → {il_raw["transaction_date"].max().date()}')
display(il_raw.head(8))

23:17:34  INFO  Fetching IL transactions for 2023…
23:17:42  INFO    1875 IL transactions found in 2023



Total IL transaction records : 1,841
Transaction type breakdown:
transaction_type
IL     880
IA     779
ITD    182

Date range: 2023-02-16 → 2023-12-27


,player_id,player_name,team,transaction_type,type_desc,transaction_date,description,season
0,425794,Adam Wainwright,St. Louis Cardinals,IL,Status Change,2023-03-30,St. Louis Cardinals placed RHP Adam Wainwright...,2023
1,425794,Adam Wainwright,St. Louis Cardinals,IA,Status Change,2023-05-06,St. Louis Cardinals activated RHP Adam Wainwri...,2023
2,425794,Adam Wainwright,St. Louis Cardinals,IL,Status Change,2023-07-05,St. Louis Cardinals placed RHP Adam Wainwright...,2023
3,425794,Adam Wainwright,St. Louis Cardinals,IA,Status Change,2023-07-24,St. Louis Cardinals activated RHP Adam Wainwri...,2023
4,425844,Zack Greinke,Kansas City Royals,IL,Status Change,2023-07-05,Kansas City Royals placed RHP Zack Greinke on ...,2023
5,425844,Zack Greinke,Kansas City Royals,IA,Status Change,2023-07-20,Kansas City Royals activated RHP Zack Greinke ...,2023
6,425844,Zack Greinke,Kansas City Royals,IL,Status Change,2023-08-08,Kansas City Royals placed RHP Zack Greinke on ...,2023
7,425844,Zack Greinke,Kansas City Royals,IA,Status Change,2023-08-22,Kansas City Royals activated RHP Zack Greinke ...,2023


## 4. Filter to Pitchers

In [21]:
before = len(il_raw)
il_pitchers = il_raw[il_raw['player_id'].isin(pitcher_ids)].copy()
after  = len(il_pitchers)

print(f'All players : {before:,} transactions')
print(f'Pitchers    : {after:,} transactions  ({after/before:.1%} of total)')
print(f'Unique pitchers with IL activity: {il_pitchers["player_id"].nunique():,}')

All players : 1,841 transactions
Pitchers    : 504 transactions  (27.4% of total)
Unique pitchers with IL activity: 183


## 5. Inspect Raw Descriptions

Before parsing, look at real description strings to confirm the keyword patterns
will work correctly.

In [22]:
placements_raw = il_pitchers[il_pitchers['transaction_type'] == 'IL'].copy()

print(f'Pitcher IL placements: {len(placements_raw):,}')
print('\nSample descriptions:')
for desc in placements_raw['description'].dropna().sample(min(10, len(placements_raw)), random_state=42):
    print(f'  {desc}')

Pitcher IL placements: 244

Sample descriptions:
  Arizona Diamondbacks placed RHP Merrill Kelly on the 15-day injured list retroactive to June 25, 2023. Right calf inflammation.
  Atlanta Braves placed RHP Jesse Chavez on the 15-day injured list. Left shin contusion.
  Oakland Athletics placed RHP Austin Pruitt on the 15-day injured list retroactive to August 17, 2023. Right forearm strain.
  Pittsburgh Pirates placed LHP Jose Hernandez on the 15-day injured list. Right calf strain.
  Washington Nationals placed RHP Mason Thompson on the 15-day injured list retroactive to August 2, 2023. Left knee contusion.
  Cleveland Guardians placed RHP Triston McKenzie on the 15-day injured list. Right elbow sprain.
  Pittsburgh Pirates placed RHP Dauri Moreta on the 15-day injured list.
  Texas Rangers placed RHP Josh Sborz on the 15-day injured list. Left hamstring strain.
  Atlanta Braves placed RHP Charlie Morton on the 15-day injured list retroactive to September 23, 2023. Right index finger

## 6. Parse Injury Types

`parse_injury_type` applies keyword patterns in priority order. The first matching
pattern wins — elbow is checked before shoulder, shoulder before forearm, etc. —
so `"right elbow strain"` maps to `elbow`, not `other`.

In [23]:
# Verify the parser on a few hand-crafted examples before applying at scale.
TEST_DESCRIPTIONS = [
    ("Placed on 15-day IL. Right elbow inflammation.",              "elbow"),
    ("Transferred to 60-day IL. Left shoulder rotator cuff tear.",  "shoulder"),
    ("Placed on 15-day IL. Right forearm strain.",                  "forearm"),
    ("Placed on 15-day IL. Right oblique strain.",                  "oblique"),
    ("Placed on 10-day IL. Left hamstring tightness.",              "hamstring"),
    ("Placed on 15-day IL. Right knee inflammation.",               "knee"),
    ("Placed on 10-day IL. Right hand blister.",                    "finger_hand"),
    ("Placed on 10-day IL. Illness.",                               "illness"),
    ("Placed on 15-day IL. Lower back tightness.",                  "back"),
]

print('Parser unit tests:')
all_pass = True
for desc, expected in TEST_DESCRIPTIONS:
    result = parse_injury_type(desc)
    status = '✓' if result == expected else '✗'
    if result != expected:
        all_pass = False
    print(f'  {status}  expected={expected:<12}  got={result:<12}  |  {desc[:60]}')

print(f'\nAll tests passed: {all_pass}')

Parser unit tests:
  ✓  expected=elbow         got=elbow         |  Placed on 15-day IL. Right elbow inflammation.
  ✓  expected=shoulder      got=shoulder      |  Transferred to 60-day IL. Left shoulder rotator cuff tear.
  ✓  expected=forearm       got=forearm       |  Placed on 15-day IL. Right forearm strain.
  ✓  expected=oblique       got=oblique       |  Placed on 15-day IL. Right oblique strain.
  ✓  expected=hamstring     got=hamstring     |  Placed on 10-day IL. Left hamstring tightness.
  ✓  expected=knee          got=knee          |  Placed on 15-day IL. Right knee inflammation.
  ✓  expected=finger_hand   got=finger_hand   |  Placed on 10-day IL. Right hand blister.
  ✓  expected=illness       got=illness       |  Placed on 10-day IL. Illness.
  ✓  expected=back          got=back          |  Placed on 15-day IL. Lower back tightness.

All tests passed: True


In [24]:
# Apply to the full pitcher placement dataset
placements_raw['injury_type'] = placements_raw['description'].apply(parse_injury_type)

print('Injury type distribution (placements only):')
counts = placements_raw['injury_type'].value_counts()
pct    = (counts / counts.sum() * 100).round(1)
display(pd.DataFrame({'count': counts, 'pct': pct}))

Injury type distribution (placements only):


,count,pct
injury_type,,
other,47,19.3
shoulder,43,17.6
elbow,33,13.5
hip,20,8.2
back,18,7.4
forearm,16,6.6
oblique,15,6.1
hamstring,13,5.3
finger_hand,12,4.9


## 7. Pair Placements with Activations → Compute Days Lost

Each IL placement (typeCode `IL`) is matched to the earliest subsequent
activation (typeCode `IA`) for the same player. IL transfers (ITD — e.g.,
15-day to 60-day) are ignored for pairing because they extend an existing
stint rather than starting a new one. `days_lost` therefore spans the full
original placement to final activation, regardless of IL list changes in between.

Placements with no matching activation in any season are flagged `season_ending=True`
with `days_lost=None`.

In [25]:
injury_db = compute_days_lost(il_pitchers)

print(f'IL stints (placements paired): {len(injury_db):,}')
print(f'  With activation date  : {injury_db["activation_date"].notna().sum():,}')
print(f'  Season-ending (no act): {injury_db["season_ending"].sum():,}')
print(f'\ndays_lost distribution (non-season-ending):')
print(injury_db[injury_db["days_lost"].notna()]["days_lost"].describe().round(1).to_string())

display(injury_db.head(8))

IL stints (placements paired): 244
  With activation date  : 210
  Season-ending (no act): 34

days_lost distribution (non-season-ending):
count    210.0
mean      37.6
std       32.7
min        4.0
25%       15.0
50%       24.0
75%       51.8
max      152.0


,player_id,player_name,team,season,transaction_date,activation_date,days_lost,season_ending,injury_type,type_desc,description
0,425794,Adam Wainwright,St. Louis Cardinals,2023,2023-03-30,2023-05-06,37.0,False,hip,Status Change,St. Louis Cardinals placed RHP Adam Wainwright...
1,425794,Adam Wainwright,St. Louis Cardinals,2023,2023-07-05,2023-07-24,19.0,False,shoulder,Status Change,St. Louis Cardinals placed RHP Adam Wainwright...
2,425844,Zack Greinke,Kansas City Royals,2023,2023-07-05,2023-07-20,15.0,False,shoulder,Status Change,Kansas City Royals placed RHP Zack Greinke on ...
3,425844,Zack Greinke,Kansas City Royals,2023,2023-08-08,2023-08-22,14.0,False,elbow,Status Change,Kansas City Royals placed RHP Zack Greinke on ...
4,434378,Justin Verlander,New York Mets,2023,2023-03-31,2023-05-04,34.0,False,other,Status Change,New York Mets placed RHP Justin Verlander on t...
5,445276,Kenley Jansen,Boston Red Sox,2023,2023-09-13,2023-09-23,10.0,False,other,Status Change,Boston Red Sox placed RHP Kenley Jansen on the...
6,445926,Jesse Chavez,Atlanta Braves,2023,2023-06-15,NaT,NaN,True,other,Status Change,Atlanta Braves placed RHP Jesse Chavez on the ...
7,446372,Corey Kluber,Boston Red Sox,2023,2023-06-21,NaT,NaN,True,shoulder,Status Change,Boston Red Sox placed RHP Corey Kluber on the ...


## 8. Validation

### 8a. Known-Injury Spot Checks

Verify specific well-documented 2023 IL stints appear correctly in the database.

In [26]:
# Well-documented 2023 IL stints to spot-check.
# Justin Verlander (434378): placed on IL in 2023 with shoulder issues.
# Spencer Strider  (675911): one of the workhorses of 2023 — check for any stints.
SPOT_CHECKS = {
    434378: 'Justin Verlander',
    477132: 'Clayton Kershaw',
    605483: 'Sandy Alcantara',
}

for pid, name in SPOT_CHECKS.items():
    rows = injury_db[injury_db['player_id'] == pid]
    if len(rows) == 0:
        print(f'{name} ({pid}): no IL stints found in {TEST_YEAR if TEST_MODE else str(START_YEAR)+"-"+str(END_YEAR)}')
    else:
        print(f'{name} ({pid}): {len(rows)} stint(s)')
        print(rows[['transaction_date','activation_date','days_lost','injury_type','description']]
              .to_string(index=False))
    print()

Justin Verlander (434378): 1 stint(s)
transaction_date activation_date  days_lost injury_type                                                                                                                       description
      2023-03-31      2023-05-04       34.0       other New York Mets placed RHP Justin Verlander on the 15-day injured list retroactive to March 28, 2023. Low grade teres major strain.

Clayton Kershaw (477132): 1 stint(s)
transaction_date activation_date  days_lost injury_type                                                                                                                     description
      2023-07-03             NaT        NaN    shoulder Los Angeles Dodgers placed LHP Clayton Kershaw on the 15-day injured list retroactive to June 30, 2023. Left shoulder soreness.

Sandy Alcantara (605483): no IL stints found in 2023



### 8b. Days Lost Sanity Checks

In [27]:
paired = injury_db[injury_db['days_lost'].notna()].copy()

# Stints shorter than 10 days are suspicious — minimum IL is 10 days.
too_short = paired[paired['days_lost'] < 10]
print(f'Stints < 10 days: {len(too_short)}')
if len(too_short) > 0:
    display(too_short[['player_name','transaction_date','activation_date','days_lost','description']].head(5))

# Stints longer than 365 days are likely data errors (no activation found
# and a next-season activation was incorrectly matched).
too_long = paired[paired['days_lost'] > 365]
print(f'Stints > 365 days: {len(too_long)}')
if len(too_long) > 0:
    display(too_long[['player_name','transaction_date','activation_date','days_lost','description']].head(5))

print(f'\nMedian days lost: {paired["days_lost"].median():.0f}')
print(f'Mean days lost  : {paired["days_lost"].mean():.0f}')

Stints < 10 days: 11


,player_name,transaction_date,activation_date,days_lost,description
11,Daniel Bard,2023-09-27,2023-10-02,5.0,Colorado Rockies placed RHP Daniel Bard on the...
13,Chris Martin,2023-09-28,2023-10-02,4.0,Boston Red Sox placed RHP Chris Martin on the ...
21,Scott Alexander,2023-09-23,2023-10-02,9.0,San Francisco Giants placed LHP Scott Alexande...
23,Fernando Cruz,2023-09-01,2023-09-08,7.0,Cincinnati Reds placed RHP Fernando Cruz on th...
32,Tyler Anderson,2023-09-24,2023-10-02,8.0,Los Angeles Angels placed LHP Tyler Anderson o...


Stints > 365 days: 0

Median days lost: 24
Mean days lost  : 38


### 8c. Injury Type Distribution

In [28]:
print('Injury type distribution (all stints):')
counts = injury_db['injury_type'].value_counts()
pct    = (counts / counts.sum() * 100).round(1)
display(pd.DataFrame({'count': counts, 'pct_%': pct}))

print('\nMean days lost by injury type:')
print(
    injury_db[injury_db['days_lost'].notna()]
    .groupby('injury_type')['days_lost']
    .agg(['mean','median','count'])
    .round(1)
    .sort_values('mean', ascending=False)
    .to_string()
)

Injury type distribution (all stints):


,count,pct_%
injury_type,,
other,47,19.3
shoulder,43,17.6
elbow,33,13.5
hip,20,8.2
back,18,7.4
forearm,16,6.6
oblique,15,6.1
hamstring,13,5.3
finger_hand,12,4.9



Mean days lost by injury type:
             mean  median  count
injury_type                     
neck         61.2    66.0      5
elbow        51.6    34.0     26
shoulder     51.1    36.5     34
forearm      43.8    34.0     12
knee         35.1    15.5      8
oblique      33.6    33.0     14
hip          32.6    33.0     17
other        32.4    23.0     41
back         31.2    19.5     16
finger_hand  27.6    15.0     11
hamstring    22.3    16.0     13
calf_ankle   19.6    16.0     11
illness      17.0    17.0      2


## 9. Save

In [29]:
save_path = INJURIES_DIR / 'injury_database.parquet'
injury_db.to_parquet(save_path, index=False)
print(f'Saved {len(injury_db):,} IL stints → {save_path.resolve()}')

# Quick reload check
check = load_injury_database(str(save_path))
assert len(check) == len(injury_db), 'Row count mismatch on reload'
print('Reload check passed.')

Saved 244 IL stints → /Users/nateseluga/Pitcher-Injury-Risk/data/raw/injuries/injury_database.parquet
Reload check passed.
